## Try to merge flight data with weather

In [1]:
%pwd

'/Users/nicholasstanfield/Desktop/flight-delay/notebooks'

In [2]:
import pandas as pd

df = pd.read_csv("../data/processed/flight_data_2025.csv")
df.head()

,QUARTER,MONTH,DAY_OF_MONTH,DAY_OF_WEEK,ORIGIN,DEST,CRS_DEP_TIME,CRS_ARR_TIME,CRS_ELAPSED_TIME,DISTANCE,AIRLINE,ROUTE,DELAY
0,1,1,1,3,AUS,ORD,545,830,165.0,977.0,American Airlines Inc.,AUS-ORD,0
1,1,1,7,2,PBI,DFW,620,842,202.0,1102.0,American Airlines Inc.,PBI-DFW,0
2,1,1,26,7,LAS,DEN,920,1219,119.0,628.0,United Air Lines Inc.,LAS-DEN,0
3,1,1,17,5,ICT,ATL,1750,2105,135.0,782.0,Delta Air Lines Inc.,ICT-ATL,0
4,1,1,14,2,CHS,EWR,1930,2129,119.0,628.0,United Air Lines Inc.,CHS-EWR,0


In [3]:
codes = list(set(list(df["ORIGIN"].unique()) + list(df["DEST"].unique()))) # grab all the airports in the dataset
codes = pd.DataFrame(codes)
codes.columns = ["code"]
codes

,code
0,MOT
1,TYS
2,SWF
3,PIR
4,GGG
...,...
346,LAN
347,BFF
348,ALO
349,DSM


In [4]:
coords = pd.read_csv("../data/raw/airport_coordinates.csv") 
coords = coords[["iata_code", "latitude_deg", "longitude_deg"]]
coords

,iata_code,latitude_deg,longitude_deg
0,NaN,40.070985,-74.933689
1,NaN,38.704022,-101.473911
2,NaN,59.947733,-151.692524
3,NaN,34.864799,-86.770302
4,NaN,59.093287,-156.456699
...,...,...,...
85931,NaN,41.784354,123.496308
85932,NaN,51.894444,1.482500
85933,NaN,-11.584278,47.296389
85934,NaN,32.110587,-97.356312


In [5]:
# this data is from the following website: https://ourairports.com/help/data-dictionary.html
# it is the airports.csv

In [6]:
airports = pd.merge(codes, coords, left_on="code", right_on="iata_code", how="inner")
airports

,code,iata_code,latitude_deg,longitude_deg
0,MOT,MOT,48.258010,-101.279123
1,TYS,TYS,35.811001,-83.994003
2,SWF,SWF,41.504210,-74.108901
3,PIR,PIR,44.382702,-100.286003
4,GGG,GGG,32.383999,-94.711502
...,...,...,...,...
345,LAN,LAN,42.777582,-84.585721
346,BFF,BFF,41.874001,-103.596001
347,ALO,ALO,42.557098,-92.400299
348,DSM,DSM,41.534027,-93.656719


In [7]:
airports = airports.drop("code", axis=1)
airports.head()

,iata_code,latitude_deg,longitude_deg
0,MOT,48.258010,-101.279123
1,TYS,35.811001,-83.994003
2,SWF,41.504210,-74.108901
3,PIR,44.382702,-100.286003
4,GGG,32.383999,-94.711502


In [8]:
# Now we have the airport coord look up table we need to get the weather data
# For this we need the coords + some time constraint
# Most natural is the hour
# So we will have DEPT_LAT, DEPT_LON, DEPT_HOUR and ARR_LAT, ARR_LON, ARR_HOUR
# Then merge some weather data (to be found) on eacg row in two ways
# 1. merge on the DEPT location and hour 2. merge on ARR location and hour
# So need to organize the time in original df properly

In [9]:
# important!
# need to account for timezones and find out what CRS_DEP_TIME and CRS_ARR_TIME timezones are
# seems like they are local time so will need to timezones to the airport coords to account for time

In [10]:
from timezonefinder import TimezoneFinder
tf = TimezoneFinder()

airports["timezone"] = airports.apply(lambda row: tf.timezone_at(lat=row["latitude_deg"], lng=row["longitude_deg"]), axis=1)
airports.head() 

,iata_code,latitude_deg,longitude_deg,timezone
0,MOT,48.258010,-101.279123,America/Chicago
1,TYS,35.811001,-83.994003,America/New_York
2,SWF,41.504210,-74.108901,America/New_York
3,PIR,44.382702,-100.286003,America/Chicago
4,GGG,32.383999,-94.711502,America/Chicago


In [11]:
# now we can merge this new info into main df

In [12]:
df = pd.read_csv("../data/processed/flight_data_2025.csv")

In [13]:
len(df)

360000

In [14]:
test = df.merge(airports, left_on="ORIGIN", right_on="iata_code", how="left")
missing = test[test["timezone"].isna()]
missing["ORIGIN"].value_counts()

ORIGIN
PBI    1573
Name: count, dtype: int64

In [15]:
# the airport name has changed... need to add the original into the airport table 

In [16]:
new_airport = {
    "iata_code": "PBI",
    "latitude_deg": 26.6832,
    "longitude_deg": -80.0956,
    "timezone": "America/New_York"
}

airports.loc[len(airports)] = new_airport

In [17]:
airports

,iata_code,latitude_deg,longitude_deg,timezone
0,MOT,48.258010,-101.279123,America/Chicago
1,TYS,35.811001,-83.994003,America/New_York
2,SWF,41.504210,-74.108901,America/New_York
3,PIR,44.382702,-100.286003,America/Chicago
4,GGG,32.383999,-94.711502,America/Chicago
...,...,...,...,...
346,BFF,41.874001,-103.596001,America/Denver
347,ALO,42.557098,-92.400299,America/Chicago
348,DSM,41.534027,-93.656719,America/Chicago
349,SCK,37.893279,-121.238079,America/Los_Angeles


In [21]:
airports.to_csv("../data/raw/airport_coordinates_20260819.csv") # use for the pipeline

In [18]:
len(df.merge(airports, left_on="ORIGIN", right_on="iata_code", how="inner")) # now it works

360000

In [19]:
df = df.merge(airports, left_on="ORIGIN", right_on="iata_code", how="inner")
df = df.rename(columns={"latitude_deg":"ORIGIN_LATITUDE","longitude_deg":"ORIGIN_LONGITUDE","timezone":"ORIGIN_TIMEZONE"})
df = df.drop('iata_code', axis=1)
df.head()

,QUARTER,MONTH,DAY_OF_MONTH,DAY_OF_WEEK,ORIGIN,DEST,CRS_DEP_TIME,CRS_ARR_TIME,CRS_ELAPSED_TIME,DISTANCE,AIRLINE,ROUTE,DELAY,ORIGIN_LATITUDE,ORIGIN_LONGITUDE,ORIGIN_TIMEZONE
0,1,1,1,3,AUS,ORD,545,830,165.0,977.0,American Airlines Inc.,AUS-ORD,0,30.197535,-97.662015,America/Chicago
1,1,1,7,2,PBI,DFW,620,842,202.0,1102.0,American Airlines Inc.,PBI-DFW,0,26.683200,-80.095600,America/New_York
2,1,1,26,7,LAS,DEN,920,1219,119.0,628.0,United Air Lines Inc.,LAS-DEN,0,36.083361,-115.151817,America/Los_Angeles
3,1,1,17,5,ICT,ATL,1750,2105,135.0,782.0,Delta Air Lines Inc.,ICT-ATL,0,37.650314,-97.428583,America/Chicago
4,1,1,14,2,CHS,EWR,1930,2129,119.0,628.0,United Air Lines Inc.,CHS-EWR,0,32.896159,-80.038151,America/New_York


In [22]:
df = df.merge(airports, left_on="DEST", right_on="iata_code", how="inner")
df = df.rename(columns={"latitude_deg":"DEST_LATITUDE","longitude_deg":"DEST_LONGITUDE","timezone":"DEST_TIMEZONE"})
df = df.drop('iata_code', axis=1)

In [25]:
df["SCHEDULED_DEP_DATETIME"] = pd.to_datetime({
    "year": 2025,
    "month": df["MONTH"],
    "day": df["DAY_OF_MONTH"],
    "hour": df["CRS_DEP_TIME"] // 100,
    "minute": df["CRS_DEP_TIME"] % 100,
})

In [26]:
df["SCHEDULED_DEP_DATETIME"]

0        2025-01-01 05:45:00
1        2025-01-07 06:20:00
2        2025-01-26 09:20:00
3        2025-01-17 17:50:00
4        2025-01-14 19:30:00
                 ...        
359995   2025-12-28 17:14:00
359996   2025-12-21 09:20:00
359997   2025-12-18 12:40:00
359998   2025-12-11 14:35:00
359999   2025-12-09 07:26:00
Name: SCHEDULED_DEP_DATETIME, Length: 360000, dtype: datetime64[ns]

In [27]:
df["SCHEDULED_DEP_UTC"] = [
    dt.tz_localize(tz).tz_convert("UTC")
    for dt, tz in zip(
        df["SCHEDULED_DEP_DATETIME"],
        df["ORIGIN_TIMEZONE"]
    )
]

In [28]:
df["SCHEDULED_DEP_UTC"]

0        2025-01-01 11:45:00+00:00
1        2025-01-07 11:20:00+00:00
2        2025-01-26 17:20:00+00:00
3        2025-01-17 23:50:00+00:00
4        2025-01-15 00:30:00+00:00
                    ...           
359995   2025-12-28 22:14:00+00:00
359996   2025-12-21 14:20:00+00:00
359997   2025-12-18 17:40:00+00:00
359998   2025-12-11 20:35:00+00:00
359999   2025-12-09 15:26:00+00:00
Name: SCHEDULED_DEP_UTC, Length: 360000, dtype: datetime64[ns, UTC]

In [30]:
df["SCHEDULED_ARR_UTC"] = (
    df["SCHEDULED_DEP_UTC"]
    + pd.to_timedelta(df["CRS_ELAPSED_TIME"], unit="m")
)

df["SCHEDULED_ARR_UTC"]

0        2025-01-01 14:30:00+00:00
1        2025-01-07 14:42:00+00:00
2        2025-01-26 19:19:00+00:00
3        2025-01-18 02:05:00+00:00
4        2025-01-15 02:29:00+00:00
                    ...           
359995   2025-12-29 02:30:00+00:00
359996   2025-12-21 17:05:00+00:00
359997   2025-12-18 20:45:00+00:00
359998   2025-12-11 22:25:00+00:00
359999   2025-12-09 16:38:00+00:00
Name: SCHEDULED_ARR_UTC, Length: 360000, dtype: datetime64[ns, UTC]

In [32]:
df["DEP_WEATHER_HOUR"] = (
    df["SCHEDULED_DEP_UTC"].dt.floor("h")
)

df["ARR_WEATHER_HOUR"] = (
    df["SCHEDULED_ARR_UTC"].dt.floor("h")
)

In [33]:
df.head(2)

,QUARTER,MONTH,DAY_OF_MONTH,DAY_OF_WEEK,ORIGIN,DEST,CRS_DEP_TIME,CRS_ARR_TIME,CRS_ELAPSED_TIME,DISTANCE,...,ORIGIN_LONGITUDE,ORIGIN_TIMEZONE,DEST_LATITUDE,DEST_LONGITUDE,DEST_TIMEZONE,SCHEDULED_DEP_DATETIME,SCHEDULED_DEP_UTC,SCHEDULED_ARR_UTC,DEP_WEATHER_HOUR,ARR_WEATHER_HOUR
0,1,1,1,3,AUS,ORD,545,830,165.0,977.0,...,-97.662015,America/Chicago,41.978600,-87.904800,America/Chicago,2025-01-01 05:45:00,2025-01-01 11:45:00+00:00,2025-01-01 14:30:00+00:00,2025-01-01 11:00:00+00:00,2025-01-01 14:00:00+00:00
1,1,1,7,2,PBI,DFW,620,842,202.0,1102.0,...,-80.095600,America/New_York,32.896801,-97.038002,America/Chicago,2025-01-07 06:20:00,2025-01-07 11:20:00+00:00,2025-01-07 14:42:00+00:00,2025-01-07 11:00:00+00:00,2025-01-07 14:00:00+00:00


In [35]:
df["CRS_ELAPSED_TIME"].astype(int)

0         165
1         202
2         119
3         135
4         119
         ... 
359995    256
359996    165
359997    185
359998    110
359999     72
Name: CRS_ELAPSED_TIME, Length: 360000, dtype: int64

## Pipeline Steps

1. Create a list of all the airline codes
2. Read in the airport coordinates (with the right columns)
3. Merge the codes and coordinates
4. Add the timezone
5. Merge on ORIGIN codes
6. Merge on DEPT codes
7. Create SCHEDULED DEPT and ARR datetime columns
8. Convert these to UTC (maybe optional if you can use the timezone in the weather API
9. Floor/round to get in date hour format e.g. 2025-01-01 14:00:00+00:00

## OpenMeteo API test

In [36]:
weather_variables = [
    "temperature_2m",
    "relative_humidity_2m",
    "precipitation",
    "rain",
    "snowfall",
    "weather_code",
    "cloud_cover",
    "visibility",
    "wind_speed_10m",
    "wind_gusts_10m",
]

import requests
import pandas as pd

url = "https://historical-forecast-api.open-meteo.com/v1/forecast"

params = {
    "latitude": 26.6832,
    "longitude": -80.0956,
    "start_date": "2025-01-01",
    "end_date": "2025-12-31",
    "hourly": ",".join(weather_variables),
    "timezone": "GMT",
}

response = requests.get(url, params=params)
response.raise_for_status()

data = response.json()

In [37]:
data

{'latitude': 26.695168,
 'longitude': -80.09199,
 'generationtime_ms': 27016.503930091858,
 'utc_offset_seconds': 0,
 'timezone': 'GMT',
 'timezone_abbreviation': 'GMT',
 'elevation': 3.0,
 'hourly_units': {'time': 'iso8601',
  'temperature_2m': '°C',
  'relative_humidity_2m': '%',
  'precipitation': 'mm',
  'rain': 'mm',
  'snowfall': 'cm',
  'weather_code': 'wmo code',
  'cloud_cover': '%',
  'visibility': 'm',
  'wind_speed_10m': 'km/h',
  'wind_gusts_10m': 'km/h'},
 'hourly': {'time': ['2025-01-01T00:00',
   '2025-01-01T01:00',
   '2025-01-01T02:00',
   '2025-01-01T03:00',
   '2025-01-01T04:00',
   '2025-01-01T05:00',
   '2025-01-01T06:00',
   '2025-01-01T07:00',
   '2025-01-01T08:00',
   '2025-01-01T09:00',
   '2025-01-01T10:00',
   '2025-01-01T11:00',
   '2025-01-01T12:00',
   '2025-01-01T13:00',
   '2025-01-01T14:00',
   '2025-01-01T15:00',
   '2025-01-01T16:00',
   '2025-01-01T17:00',
   '2025-01-01T18:00',
   '2025-01-01T19:00',
   '2025-01-01T20:00',
   '2025-01-01T21:00',
  

In [38]:
pbi_weather = pd.DataFrame(data["hourly"])

pbi_weather.head()

,time,temperature_2m,relative_humidity_2m,precipitation,rain,snowfall,weather_code,cloud_cover,visibility,wind_speed_10m,wind_gusts_10m
0,2025-01-01T00:00,23.1,87,0.0,0.0,0.0,1,32,14700.0,3.1,23.0
1,2025-01-01T01:00,22.6,83,0.0,0.0,0.0,0,0,17500.0,8.5,27.4
2,2025-01-01T02:00,22.1,84,0.0,0.0,0.0,0,0,16400.0,7.5,32.8
3,2025-01-01T03:00,21.3,88,0.0,0.0,0.0,0,3,14300.0,8.6,28.4
4,2025-01-01T04:00,21.3,90,0.0,0.0,0.0,0,17,13300.0,7.3,28.8


In [39]:
pbi_weather.shape

(8760, 11)

In [40]:
365*24

8760

In [41]:
pbi_weather["time"] = pd.to_datetime(
    pbi_weather["time"],
    utc=True
)

In [42]:
pbi_weather.head()

,time,temperature_2m,relative_humidity_2m,precipitation,rain,snowfall,weather_code,cloud_cover,visibility,wind_speed_10m,wind_gusts_10m
0,2025-01-01 00:00:00+00:00,23.1,87,0.0,0.0,0.0,1,32,14700.0,3.1,23.0
1,2025-01-01 01:00:00+00:00,22.6,83,0.0,0.0,0.0,0,0,17500.0,8.5,27.4
2,2025-01-01 02:00:00+00:00,22.1,84,0.0,0.0,0.0,0,0,16400.0,7.5,32.8
3,2025-01-01 03:00:00+00:00,21.3,88,0.0,0.0,0.0,0,3,14300.0,8.6,28.4
4,2025-01-01 04:00:00+00:00,21.3,90,0.0,0.0,0.0,0,17,13300.0,7.3,28.8


In [43]:
# TODO LOOK AT METEO OPEN API  https://open-meteo.com/en/docs/historical-weather-api